# 🫁 LUNGRAY DATA PREPARATION: NIH CHESTX-RAY14 512x512 HIGH-SPEED RESCALING ENGINE
## Chuẩn Hóa 112,120 Ảnh NIH Về Chuẩn $512 \times 512$ Bằng Thuật Toán `cv2.INTER_AREA` Đa Luồng Trên Kaggle Cloud

---

### 🎯 Mục tiêu kỹ thuật:
1. **0 Byte ổ cứng local**: Thực thi 100% trên Kaggle Cloud, không cần tải 45GB ảnh thô về máy cá nhân.
2. **Bảo tồn vi tổn thương học**: Sử dụng thuật toán `cv2.INTER_AREA` chuẩn công nghiệp để chống hiện tượng răng cưa và giữ nguyên vẹn các nốt mờ nhỏ, xơ hóa và đường kẻ giải phẫu.
3. **Đa tiến trình tối ưu**: Tận dụng 4 vCPUs của Kaggle (`concurrent.futures.ProcessPoolExecutor`) để hoàn tất 112,120 ảnh trong ~25 phút.
4. **Tạo Dataset `lungray-nih-512`**: Xuất thư mục ảnh 512x512 (~12-14 GB, an toàn dưới giới hạn 20GB của `/kaggle/working`) kèm `Data_Entry_2017.csv` và `patient_splits.csv` để sẵn sàng huấn luyện tốc độ cao.

### 📌 Hướng dẫn thêm Data trên Kaggle:
- Bấm **+ Add Data** ở góc phải màn hình Notebook.
- Tìm kiếm và thêm dataset: **`nih-chest-xrays`** (hoặc `nih-chest-xrays/data`).

In [1]:
# CELL 1: Kiểm tra môi trường Kaggle & Dung lượng đĩa trống
import os
import sys
import shutil
import time
import glob

cpu_cores = os.cpu_count() or 4
total_disk, used_disk, free_disk = shutil.disk_usage('/kaggle/working' if os.path.exists('/kaggle') else '.')

print('=' * 60)
print(f'[*] Số nhân CPU khả dụng: {cpu_cores} vCPUs')
print(f'[*] Dung lượng ổ đĩa tổng: {total_disk / (1024**3):.2f} GB')
print(f'[*] Dung lượng ổ đĩa còn trống: {free_disk / (1024**3):.2f} GB')
print('=' * 60)
assert free_disk / (1024**3) >= 15.0, '[CẢNH BÁO] Dung lượng đĩa trống dưới 15GB! Hãy dọn dẹp /kaggle/working trước khi chạy.'

[*] Số nhân CPU khả dụng: 4 vCPUs
[*] Dung lượng ổ đĩa tổng: 19.52 GB
[*] Dung lượng ổ đĩa còn trống: 19.50 GB


In [2]:
# CELL 2: Tự động phát hiện đường dẫn các chunk ảnh NIH
potential_nih_dirs = [
    '/kaggle/input/datasets/nih-chest-xrays/data',
    '/kaggle/input/datasets/nih-chest-xrays',
    '/kaggle/input/nih-chest-xrays/data',
    '/kaggle/input/nih-chest-xrays',
    '/kaggle/input/data',
    'dataset/nih-chestXray-14',
    'dataset/nih-chestXray-14/data'
]

nih_root = None
for cand in potential_nih_dirs:
    if os.path.exists(cand):
        chunks = glob.glob(os.path.join(cand, 'images_*', 'images'))
        if not chunks:
            chunks = glob.glob(os.path.join(cand, '*', 'images_*', 'images'))
        if not chunks and os.path.exists(os.path.join(cand, 'images')):
            chunks = [os.path.join(cand, 'images')]
        if chunks:
            nih_root = cand
            break

if nih_root is None:
    raise FileNotFoundError("[LỖI] Không tìm thấy thư mục ảnh NIH! Hãy kiểm tra dataset 'nih-chest-xrays'.")

print(f'[OK] Đã định vị thư mục gốc NIH: {nih_root}')

# Quét danh sách toàn bộ ảnh PNG
print('[*] Đang quét danh sách toàn bộ ảnh PNG trong các chunks...')
all_image_paths = []
for root, _, files in os.walk(nih_root):
    for f in files:
        if f.lower().endswith('.png'):
            all_image_paths.append(os.path.join(root, f))

print(f'[OK] Tìm thấy tổng cộng: {len(all_image_paths):,} ảnh PNG gốc.')


FileNotFoundError: [LỖI] Không tìm thấy thư mục ảnh NIH! Hãy đảm bảo đã Add Data 'nih-chest-xrays'.

In [ ]:
# CELL 3: Định vị metadata Data_Entry_2017.csv
import pandas as pd
from sklearn.model_selection import train_test_split

csv_candidates = [
    os.path.join(nih_root, 'Data_Entry_2017.csv'),
    os.path.join(os.path.dirname(nih_root), 'Data_Entry_2017.csv'),
    '/kaggle/input/datasets/nih-chest-xrays/Data_Entry_2017.csv',
    '/kaggle/input/nih-chest-xrays/Data_Entry_2017.csv',
    'dataset/nih-chestXray-14/Data_Entry_2017.csv'
]
csv_path = next((c for c in csv_candidates if os.path.exists(c)), None)
print(f'[*] Metadata CSV: {csv_path}')

if csv_path:
    df_meta = pd.read_csv(csv_path)
    print(f'[OK] Nạp thành công {len(df_meta):,} dòng metadata từ Data_Entry_2017.csv.')
    print(f'[*] Số lượng bệnh nhân độc nhất: {df_meta["Patient ID"].nunique():,}')
else:
    df_meta = None
    print('[CẢNH BÁO] Không tìm thấy Data_Entry_2017.csv.')


In [ ]:
# CELL 4: Khảo sát 3 ảnh gốc trước khi resize (Độ phân giải 1024x1024)
import cv2
import matplotlib.pyplot as plt

sample_paths = all_image_paths[:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, path in zip(axes, sample_paths):
    img_bgr = cv2.imread(path)
    h, w, c = img_bgr.shape
    size_kb = os.path.getsize(path) / 1024
    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{os.path.basename(path)}\n{w}x{h} ({size_kb:.1f} KB)', fontsize=10)
    ax.axis('off')

plt.suptitle('Khảo sát Mẫu Ảnh Gốc NIH ChestX-ray14 (1024x1024)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 5: Hàm xử lý đơn ảnh với cv2.INTER_AREA (Tối ưu downsampling y tế)
TARGET_SIZE = (512, 512)
PNG_COMPRESSION = 4  # Mức nén cân bằng tốc độ và dung lượng

def resize_and_save_nih(job):
    src_path, dst_path = job
    if os.path.exists(dst_path):
        return True
    try:
        img = cv2.imread(src_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            return False
        # cv2.INTER_AREA chống răng cưa và bảo tồn tổn thương vi mô
        resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        cv2.imwrite(dst_path, resized, [cv2.IMWRITE_PNG_COMPRESSION, PNG_COMPRESSION])
        return True
    except Exception:
        return False

In [ ]:
# CELL 6: Thực thi song song đa tiến trình (ProcessPoolExecutor)
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

output_root = '/kaggle/working/nih_512' if os.path.exists('/kaggle') else 'data/processed/nih_512'
output_images_dir = os.path.join(output_root, 'images')
os.makedirs(output_images_dir, exist_ok=True)

jobs = []
for src in all_image_paths:
    dst = os.path.join(output_images_dir, os.path.basename(src))
    jobs.append((src, dst))

print(f'[*] Chuẩn bị khởi chạy {len(jobs):,} tác vụ resize trên {cpu_cores} nhân CPU...')
start_time = time.time()
success_count = 0

with ProcessPoolExecutor(max_workers=cpu_cores) as executor:
    for res in tqdm(executor.map(resize_and_save_nih, jobs, chunksize=100), total=len(jobs), desc='Rescaling NIH 512x512'):
        if res:
            success_count += 1

elapsed = time.time() - start_time
print(f'\n[OK] Hoàn tất quá trình resize sau {elapsed/60:.2f} phút ({len(jobs)/max(elapsed, 1):.1f} ảnh/giây)!')
print(f'[OK] Đã ghi thành công: {success_count:,} / {len(jobs):,} ảnh.')

In [ ]:
# CELL 7: Trực quan hóa đối chứng Before vs After & Kiểm toán độ phân giải 512x512
sample_out = glob.glob(os.path.join(output_images_dir, '*.png'))[:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, path in zip(axes, sample_out):
    img_bgr = cv2.imread(path)
    h, w, c = img_bgr.shape
    size_kb = os.path.getsize(path) / 1024
    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{os.path.basename(path)}\n{w}x{h} ({size_kb:.1f} KB)', fontsize=10)
    ax.axis('off')

plt.suptitle('Kiểm toán Kết quả NIH Sau Khi Resize (Chính xác 512x512)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 8: Sao chép Metadata & Tạo patient_splits.csv (Zero Patient Leakage)
if csv_path and os.path.exists(csv_path):
    shutil.copy2(csv_path, os.path.join(output_root, 'Data_Entry_2017.csv'))
    print('[OK] Đã sao chép Data_Entry_2017.csv vào thư mục output.')

# Tạo file splits chuẩn 70/10/20 theo Patient ID
splits_out_path = os.path.join(output_root, 'patient_splits.csv')
if df_meta is not None:
    unique_patients = df_meta['Patient ID'].unique()
    train_pts, temp_pts = train_test_split(unique_patients, test_size=0.30, random_state=42)
    val_pts, test_pts = train_test_split(temp_pts, test_size=0.6667, random_state=42)
    
    train_set = set(train_pts)
    val_set = set(val_pts)
    test_set = set(test_pts)
    
    split_col = []
    for pid in df_meta['Patient ID']:
        if pid in train_set:
            split_col.append('train')
        elif pid in val_set:
            split_col.append('val')
        else:
            split_col.append('test')
            
    df_meta['split'] = split_col
    df_meta.to_csv(splits_out_path, index=False)
    print(f'[OK] Đã tạo patient_splits.csv tại: {splits_out_path}')
    print('[*] Phân bố mẫu:', df_meta['split'].value_counts().to_dict())
    assert len(train_set.intersection(test_set)) == 0, 'LỖI DATA LEAKAGE!'
    print('[CHỨNG NHẬN] ZERO PATIENT DATA LEAKAGE: Train cap Test = Rong 100%!')

In [ ]:
# CELL 9: Đo tổng dung lượng đầu ra và kiểm toán an toàn ổ đĩa
output_files = os.listdir(output_images_dir)
total_bytes = sum(os.path.getsize(os.path.join(output_images_dir, f)) for f in output_files)
total_gb = total_bytes / (1024**3)

print('=' * 60)
print('🫁 TỔNG KẾT BỘ DỮ LIỆU NIH 512x512 MỚI:')
print(f'- Tổng số ảnh: {len(output_files):,} ảnh')
print(f'- Tổng dung lượng: {total_gb:.2f} GB')
print(f'- Thư mục output: {output_root}')
print('=' * 60)
if total_gb < 18.0:
    print('[HOÀN TOÀN AN TOÀN] Dung lượng nằm gọn trong giới hạn 20GB của Kaggle Working!')
else:
    print('[CẢNH BÁO] Dung lượng gần chạm ngưỡng 20GB!')

---

## 🚀 HƯỚNG DẪN 1-CLICK TẠO KAGGLE DATASET `lungray-nih-512`

Sau khi notebook chạy xong các cell trên:
1. Ở góc trên bên phải màn hình notebook, bấm **Save Version** $\to$ Chọn **Save & Run All (Commit)** $\to$ Bấm **Save**.
2. Sau khi commit hoàn tất, vào tab **Output** ở thanh bên phải (hoặc trang xem notebook).
3. Tại mục `nih_512`, bấm dấu **3 chấm (...)** $\to$ Chọn **Create New Dataset**.
4. Đặt tiêu đề dataset: **`lungray-nih-512`** và bấm **Create**.
5. Dataset `lungray-nih-512` này giờ đây có thể được Add thẳng vào notebook huấn luyện `02_multitask_baseline_training.ipynb` để huấn luyện tốc độ cao với **GPU load 98%**!